# 10 — Daten zurücksetzen und Infrastruktur herunterfahren

## Zweck
Den Projektzustand sauber zurücksetzen: alle von den Notebooks erzeugten Dateien unter `data/` löschen
und — lokal — die Docker-Infrastruktur herunterfahren. Danach kann die Pipeline (Notebooks `00`–`09`)
aus einem sauberen Zustand neu starten.

Geschützt bleiben `.gitkeep`-Dateien (Verzeichnisstruktur) und der Ordner `data/test-executed-notebooks/`.
Alle gelöschten Daten lassen sich durch erneutes Ausführen der Pipeline wiederherstellen.

## Konfiguration

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import subprocess

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)
DATA_DIR = PROJECT_ROOT / "data"
EXECUTION_ENV = os.getenv("EXECUTION_ENV", "docker_compose")

PROTECTED_NAMES = {".gitkeep"}
PROTECTED_DIR = "test-executed-notebooks"
print({"execution_env": EXECUTION_ENV, "data_dir": str(DATA_DIR)})

## Generierte Dateien löschen

In [ ]:
deleted = 0
failed = []
for path in DATA_DIR.rglob("*"):
    if not path.is_file() or path.name in PROTECTED_NAMES:
        continue
    if PROTECTED_DIR in {p.name for p in path.parents}:
        continue
    try:
        path.unlink()
        deleted += 1
    except OSError as exc:
        # z.B. gesperrte Datei (offenes Parquet/Checkpoint) -> weitermachen, am Ende melden
        failed.append((str(path.relative_to(PROJECT_ROOT)), str(exc)))

# Leere, generierte Verzeichnisse von innen nach aussen entfernen.
# Geschuetzte Struktur bleibt erhalten: test-executed-notebooks/ und Ordner mit .gitkeep
# (deren .gitkeep nie geloescht wird -> Ordner ist nie leer -> rmdir schlaegt fehl -> bleibt).
removed_dirs = 0
for path in sorted(DATA_DIR.rglob("*"), key=lambda p: len(p.parts), reverse=True):
    if not path.is_dir():
        continue
    if path.name == PROTECTED_DIR or PROTECTED_DIR in {p.name for p in path.parents}:
        continue
    try:
        path.rmdir()          # entfernt nur, wenn leer
        removed_dirs += 1
    except OSError:
        pass                  # nicht leer -> behalten

print(f"{deleted} Dateien geloescht, {removed_dirs} leere Verzeichnisse entfernt.")
if failed:
    print(f"WARNUNG: {len(failed)} Datei(en) konnten NICHT geloescht werden (evtl. noch geoeffnet):")
    for name, exc in failed:
        print(f"  - {name}: {exc}")

## Infrastruktur herunterfahren (nur lokal, falls vorhanden)
Nur in der lokalen Docker-Umgebung (`EXECUTION_ENV=docker_compose`) und nur wenn Docker installiert ist,
stoppt `docker compose down` Kafka, Spark, PostgreSQL und Jupyter. Die PostgreSQL-Daten im Volume bleiben
erhalten; für einen vollständigen Reset `docker compose down -v` im Terminal verwenden.

In der **FH-Umgebung** gibt es keine eigene Infrastruktur — dort werden ausschließlich die generierten
Daten gelöscht. Ist Docker nicht vorhanden oder der Stack bereits gestoppt, wird der Schritt übersprungen,
ohne das Notebook abzubrechen.

In [ ]:
import shutil

if EXECUTION_ENV != "docker_compose":
    # FH-Umgebung: keine eigene Infrastruktur -> nur Daten wurden geloescht.
    print("FH-Umgebung: keine lokale Infrastruktur zum Herunterfahren (nur Daten geloescht).")
elif shutil.which("docker") is None:
    # "Falls vorhanden": kein Docker installiert -> ueberspringen, nicht abbrechen.
    print("Docker nicht gefunden -> 'docker compose down' uebersprungen.")
else:
    result = subprocess.run(
        ["docker", "compose", "down"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print("Infrastruktur heruntergefahren (docker compose down).")
    else:
        # z.B. Stack bereits gestoppt oder Daemon nicht erreichbar -> melden, nicht crashen.
        print("Hinweis: 'docker compose down' meldete einen Fehler (evtl. schon gestoppt):")
        print((result.stderr or result.stdout).strip())

## Fertig
Der Projektzustand ist zurückgesetzt. Für einen neuen Lauf mit Notebook `00` beginnen.